# Vector store

## Chroma DB
**pip install chromadb

## Getting requirements like documents, embeddings

### Getting embeddings

In [ ]:
text = "Hello world"+"what's going on"

from langchain_ollama import OllamaEmbeddings
embeder = OllamaEmbeddings(model="nomic-embed-text:latest")
vector = embeder.embed_query(text) # It takes strings as input
print(vector)

[-0.012739737, 0.014173404, -0.17213649, 0.05888316, 0.06612349, 0.059363212, 0.0021445511, -0.05463422, -0.030660806, -0.09845523, -0.016396565, 0.0716736, 0.033765644, 0.034708124, 0.045679867, -0.060925093, -0.04983222, -0.0115896, -0.06978597, 0.013320741, -0.046197753, -0.021857504, -0.009883402, 0.025226563, 0.07892625, 0.033543646, 0.0097224265, 0.02561195, 0.00486671, -0.0056971023, 0.006960165, -0.04996788, -0.032676697, -0.00852581, -0.04069036, -0.011366447, 0.091648005, -0.00789728, 0.061102983, -0.04146365, -0.021071237, 0.02749852, 0.025797544, -0.009444513, 0.029052265, -0.03104731, 0.008391294, -0.033465914, 0.0061894753, -0.06565068, -0.067666315, 0.021503067, 0.017956493, 0.034611586, 0.071852975, -0.0007540254, 0.052982584, -0.013091771, -0.014342862, 0.040792044, 0.04163994, 0.00987946, 0.004939939, 0.027010763, -0.010593901, -0.09790457, -0.029824894, 0.024144009, 0.023904206, -0.034112513, 0.09372641, -0.043848548, 0.01536265, 0.020517314, -0.028348314, -0.0545759

### Try at normal list of strings

In [16]:
#from langchain_vectorstore import Chroma
from langchain_chroma import Chroma

text = ["Hello world","what's going on"]
#text = "Hello world what's up"
ch_text_db = Chroma.from_texts(texts=text,embedding=embeder,collection_name="my_text", persist_directory="chroma_db")
#ch_text_db.persist()
print(ch_text_db.get()["documents"])

['Hello world', "what's going on"]


### Try at the book paper.pdf

In [17]:
from langchain_community.document_loaders import PyPDFLoader

path = "paper.pdf"
pdf = PyPDFLoader(path).load()
print(pdf[1].page_content[:200])

CNN has higher detection accuracy once trained on large 
datasets. In case if the Stage 1 was unable to detect any intruder, 
our system utilizes cascading classifier to further detect any 
potential 


### Split the pdf into list

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_split = RecursiveCharacterTextSplitter(
    chunk_size=100, chunk_overlap=9
).split_text(pdf[1].page_content)

print(len(recursive_split))
print(type(recursive_split))
print(recursive_split[:200])

92
<class 'list'>
['CNN has higher detection accuracy once trained on large', 'datasets. In case if the Stage 1 was unable to detect any intruder,', 'our system utilizes cascading classifier to further detect any', 'potential intruders in Stage 2. At this stage, the algorithm', 'transforms images to greyscale and uses Local Binary Pattern', '(LBP) algorithm for fast identification. The cascading classifier', 'is used to detect any undetected intruder from the previous stage.', 'If Stage 2 detected any intruder, trained cascading face detector', 'and recognizer can identify intruders using principal component \nanalysis (PCA) in Stage 3. \nIII. C', 'III. C\nONVOLUTIONAL NEURAL NETWORK FOR TRESPASSING \nDETECTION', 'In Stage 1 of SIDSS, we trained a multilayer convolutional', 'neural network (CNN) with real surveillance images of large', 'datasets to predict multiple suspicious objects. For training, we', 'created new datasets for SIDSS by collecting approximately', '3,000 surveillance i

### Vectorise the splitter data??? no need it can store and vectorise directly
## Store data to ChromaDB

#### Clear DB first

In [45]:
import shutil, os, gc

# Step 1: Release any open ChromaDB connections still in memory
try:
    del text_db
except: pass
try:
    del ch_text_db
except: pass
gc.collect()   # force Python to release file handles

# Step 2: Now delete the folder
db_path = "chroma_db"
if os.path.exists(db_path):
    # Fix permissions first, THEN delete
    for root, dirs, files in os.walk(db_path):
        for f in files:
            os.chmod(os.path.join(root, f), 0o777)
        for d in dirs:
            os.chmod(os.path.join(root, d), 0o777)
    shutil.rmtree(db_path)
    print("Deleted chroma_db")
else:
    print("Already clean")

Already clean


In [46]:
metadatas = [{"source": "paper.pdf", "page": 1} for _ in recursive_split]

text_db = Chroma.from_texts(
    texts=recursive_split,
    embedding=embeder,
    metadatas=metadatas,
    collection_name="paper",
    persist_directory="chroma_db"
)
print("Done:", text_db.get()["documents"][:2])

InternalError: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database

### Add more texts to ChromaDB

In [33]:
text_db.add_texts(texts=["ohio gujaimos","hehehe"])
print(text_db.get()["documents"][-5:])

['hehehe', 'ohio gujaimos', 'hehehe', 'ohio gujaimos', 'hehehe']


## Similarity search

In [36]:
result = text_db.similarity_search("vision",k=4)
print(result)

for x in result:
    print(x.page_content, x.metadata)

[Document(id='c976bb7e-d08c-4796-982b-3be094547c9d', metadata={}, page_content='input images, enhancing the spatial invariance of our CNN. \nBatch \nNormalization'), Document(id='3f9a73af-4024-4ebe-b680-94d65d5746b5', metadata={}, page_content='input images, enhancing the spatial invariance of our CNN. \nBatch \nNormalization'), Document(id='435178dd-2025-4509-a23d-51f1ec894e05', metadata={}, page_content='input images, enhancing the spatial invariance of our CNN. \nBatch \nNormalization'), Document(id='986d4ec1-5528-473e-8835-e95639ec07c0', metadata={}, page_content='input images, enhancing the spatial invariance of our CNN. \nBatch \nNormalization')]
input images, enhancing the spatial invariance of our CNN. 
Batch 
Normalization {}
input images, enhancing the spatial invariance of our CNN. 
Batch 
Normalization {}
input images, enhancing the spatial invariance of our CNN. 
Batch 
Normalization {}
input images, enhancing the spatial invariance of our CNN. 
Batch 
Normalization {}
